# Task 03 — Feature Engineering & Preprocessing Pipeline
**RabTech Academy · Artificial Intelligence & Machine Learning Internship**

**Goal:** build a robust, *leak-free*, reusable scikit-learn preprocessing pipeline on a complex tabular dataset
(customer churn, the same dataset used in Task 02), covering feature encoding, scaling, missing-value imputation,
correlation analysis and feature-importance analysis.

| Requirement | Where it is done |
|---|---|
| Load a complex tabular dataset | Section 1 |
| `ColumnTransformer` + `Pipeline` for numeric scaling and categorical One-Hot encoding | Section 4 |
| Train/test split **before** any transformation (no data leakage) | Section 3 (+ proof in Section 5) |
| Feature-importance analysis (mutual information and tree-based) | Section 7 |
| Missing-value imputation and correlation analysis | Sections 2, 4 and 6 |
| Reusable Scikit-Learn Pipeline code | Section 8 |

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
print("Setup complete")

## 1. Load a complex tabular dataset
Set `DATA_PATH` to your customer-churn training CSV (the one from Task 02) and `TARGET` to its label column.
If `TARGET` is left as `None`, the notebook looks for a column named like *churn*.

> If the CSV is not found, the notebook falls back to a **synthetic churn dataset** (mixed numeric/categorical columns,
> missing values, an ID column and a correlated pair of features) so that every cell still runs end to end.
> Replace it with the real file before submitting.

In [ ]:
DATA_PATH = "customer_churn_train.csv"   # <-- put your CSV path here
TARGET = None                              # e.g. "Churn"; None = auto-detect

def make_synthetic_churn(n=5000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    tenure = rng.integers(0, 73, n)
    monthly = np.round(rng.normal(65, 28, n).clip(18, 120), 2)
    total = np.round(monthly * tenure + rng.normal(0, 40, n), 2).clip(0)
    contract = rng.choice(["Month-to-month", "One year", "Two year"], n, p=[.55, .25, .20])
    internet = rng.choice(["Fiber optic", "DSL", "No"], n, p=[.44, .34, .22])
    payment = rng.choice(["Electronic check", "Mailed check", "Bank transfer", "Credit card"], n)
    gender = rng.choice(["Male", "Female"], n)
    senior = rng.choice([0, 1], n, p=[.84, .16])
    tech = rng.choice(["Yes", "No"], n)
    logit = (-1.2 + 1.1 * (contract == "Month-to-month") - 0.03 * tenure + 0.012 * monthly
             + 0.5 * (internet == "Fiber optic") + 0.35 * (payment == "Electronic check")
             + 0.3 * senior - 0.4 * (tech == "Yes"))
    churn = np.where(rng.random(n) < 1 / (1 + np.exp(-logit)), "Yes", "No")
    df = pd.DataFrame({"customerID": [f"C{i:05d}" for i in range(n)], "gender": gender,
                       "SeniorCitizen": senior, "tenure": tenure, "Contract": contract,
                       "InternetService": internet, "PaymentMethod": payment, "TechSupport": tech,
                       "MonthlyCharges": monthly, "TotalCharges": total, "Churn": churn})
    for col, frac in [("TotalCharges", .03), ("MonthlyCharges", .02), ("Contract", .02), ("TechSupport", .03)]:
        df.loc[rng.random(n) < frac, col] = np.nan
    return df

if Path(DATA_PATH).exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded real dataset: {DATA_PATH}")
else:
    df = make_synthetic_churn()
    print(f"'{DATA_PATH}' not found -> using SYNTHETIC churn data (replace with the real CSV).")

if TARGET is None:
    TARGET = next(c for c in df.columns if "churn" in c.lower())
print("Target column:", TARGET)
print("Shape:", df.shape)
df.head()

## 2. Initial data audit (missing values, types, target balance)

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
display(audit)
print("\nTarget distribution:")
print(df[TARGET].value_counts(normalize=True).round(3))

In [ ]:
# Light, *stateless* clean-up that is safe to do before the split
# (it uses no statistics learned from the data, so it cannot leak).
df = df.copy()

# 1) Drop identifier-like columns: unique per row, no predictive meaning
id_cols = [c for c in df.columns if c != TARGET and df[c].nunique() == len(df)]
print("Dropping ID-like columns:", id_cols)
df = df.drop(columns=id_cols)

# 2) Numbers stored as text (e.g. blank strings in TotalCharges) -> numeric NaN
for c in df.select_dtypes(include="object").columns.drop(TARGET, errors="ignore"):
    converted = pd.to_numeric(df[c], errors="coerce")
    if converted.notna().sum() >= 0.95 * df[c].notna().sum() and df[c].notna().sum() > 0:
        df[c] = converted
        print(f"Converted '{c}' to numeric")

# 3) Encode target to 0/1 (the label is not a feature; no leakage risk)
positive = sorted(df[TARGET].dropna().unique(), key=str)[-1]   # e.g. "Yes" / 1
y = (df[TARGET] == positive).astype(int)
X = df.drop(columns=[TARGET])
print(f"Positive class = {positive!r}; churn rate = {y.mean():.3f}")

## 3. Train/test split — *before* any transformation
Imputation medians, scaler means/standard deviations and One-Hot categories are all **learned from data**.
If they were learned on the full dataset, information from the test set would leak into training.
So we split first (stratified, to keep the churn rate identical in both parts) and fit every transformer on the training set only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate  train: %.3f  test: %.3f" % (y_train.mean(), y_test.mean()))

## 4. Preprocessing with `ColumnTransformer` + `Pipeline`
* **Numeric:** median imputation → standard scaling
* **Categorical:** most-frequent imputation → One-Hot encoding (`handle_unknown="ignore"` so unseen categories at prediction time never crash the pipeline)

In [ ]:
def build_preprocessor(X_ref: pd.DataFrame) -> ColumnTransformer:
    """Reusable, leak-free preprocessor. Column types are read from X_ref only
    (names/dtypes, never values), so it can be reused for any tabular dataset."""
    numeric_cols = X_ref.select_dtypes(include="number").columns.tolist()
    categorical_cols = X_ref.select_dtypes(exclude="number").columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

preprocessor = build_preprocessor(X_train)
print("Numeric    :", preprocessor.transformers[0][2])
print("Categorical:", preprocessor.transformers[1][2])
preprocessor

## 5. Fit on train only, transform both — and prove there is no leakage

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)   # learns medians/means/std/categories from TRAIN only
X_test_t = preprocessor.transform(X_test)         # only applies what was learned

feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_t, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_t, columns=feature_names, index=X_test.index)

print("Transformed train:", X_train_df.shape, "| test:", X_test_df.shape)
print("Missing values after pipeline  train:", int(X_train_df.isna().sum().sum()),
      "| test:", int(X_test_df.isna().sum().sum()))

# --- Leakage proof ---------------------------------------------------------
num_cols = preprocessor.transformers_[0][2]
scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]
imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]
train_imputed_mean = pd.DataFrame(imputer.transform(X_train[num_cols]), columns=num_cols).mean().values
assert np.allclose(scaler.mean_, train_imputed_mean), "Scaler was not fitted on train only!"
print("\nScaler mean_ equals the TRAIN mean  -> fitted on training data only: OK")
print("Train numeric means ~0:", X_train_df[num_cols].mean().round(3).to_dict())
print("Test  numeric means (not exactly 0, as expected):", X_test_df[num_cols].mean().round(3).to_dict())
X_train_df.head()

## 6. Correlation analysis (training data only)

In [ ]:
corr = X_train_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=45, ha="right")
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im); ax.set_title("Correlation of numeric features (train)")
plt.tight_layout(); plt.show()

# Highly correlated pairs (candidates for redundancy / multicollinearity)
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().rename("corr").reset_index())
pairs.columns = ["feature_1", "feature_2", "corr"]
high = pairs[pairs["corr"].abs() >= 0.7].sort_values("corr", key=abs, ascending=False)
print("Pairs with |corr| >= 0.7:")
display(high if len(high) else "None")

# Correlation of every encoded feature with the target
target_corr = X_train_df.corrwith(y_train).sort_values(key=abs, ascending=False)
print("Top 10 features by |correlation| with the target:")
display(target_corr.head(10).round(3).to_frame("corr_with_target"))

## 7. Feature-importance analysis
Two complementary views, both computed on the **training set only**:
1. **Mutual information** — model-free, captures non-linear dependence between each feature and the target.
2. **Random-forest (tree-based) importance** — importance as used by a non-linear model.

In [ ]:
# 7.1 Mutual information
is_discrete = np.array([f not in num_cols for f in feature_names])    # one-hot columns are discrete
mi = mutual_info_classif(X_train_df.values, y_train, discrete_features=is_discrete, random_state=RANDOM_STATE)
mi_s = pd.Series(mi, index=feature_names).sort_values(ascending=False)

# 7.2 Tree-based importance
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1,
                            class_weight="balanced", random_state=RANDOM_STATE)
rf.fit(X_train_df, y_train)
rf_s = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

TOP = min(15, len(feature_names))
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
mi_s.head(TOP)[::-1].plot.barh(ax=axes[0], color="#E0A800"); axes[0].set_title("Mutual information (top %d)" % TOP)
rf_s.head(TOP)[::-1].plot.barh(ax=axes[1], color="#2E7D32"); axes[1].set_title("Random-forest importance (top %d)" % TOP)
plt.tight_layout(); plt.show()

ranking = pd.DataFrame({"mutual_info": mi_s, "rf_importance": rf_s})
ranking["mi_rank"] = ranking["mutual_info"].rank(ascending=False).astype(int)
ranking["rf_rank"] = ranking["rf_importance"].rank(ascending=False).astype(int)
ranking = ranking.sort_values("rf_rank")
display(ranking.head(TOP).round(4))

agree = set(mi_s.head(10).index) & set(rf_s.head(10).index)
print(f"\nFeatures in BOTH top-10 lists ({len(agree)}): {sorted(agree)}")
low = ranking[(ranking.mutual_info < ranking.mutual_info.quantile(.25)) &
              (ranking.rf_importance < ranking.rf_importance.quantile(.25))].index.tolist()
print(f"Low-value candidates for removal (bottom quartile in both): {low}")

## 8. Reusable end-to-end pipeline (preprocessing + model)
Preprocessing and model are chained in one `Pipeline`, so cross-validation re-fits the imputer, scaler and encoder
inside every fold — leakage-free by construction — and the whole object can be saved and reused.

In [ ]:
def make_full_pipeline(X_ref, model=None):
    if model is None:
        model = LogisticRegression(max_iter=1000, class_weight="balanced")
    return Pipeline([("preprocess", build_preprocessor(X_ref)), ("model", model)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidates = {
    "LogisticRegression": make_full_pipeline(X_train),
    "RandomForest": make_full_pipeline(X_train, RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)),
}

for name, pipe in candidates.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name:20s} 5-fold CV ROC-AUC (train only): {scores.mean():.3f} ± {scores.std():.3f}")

# Final check on the untouched test set
final = candidates["LogisticRegression"].fit(X_train, y_train)
proba = final.predict_proba(X_test)[:, 1]
print("\nTest ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, final.predict(X_test), digits=3))

In [ ]:
# Persist the fitted pipeline so it can be re-used on new raw data
import joblib
joblib.dump(final, "churn_preprocessing_model_pipeline.joblib")

# Demo: the saved pipeline accepts RAW rows (with NaNs and even unseen categories)
reloaded = joblib.load("churn_preprocessing_model_pipeline.joblib")
sample = X_test.head(3).copy()
sample.iloc[0, sample.columns.get_loc(sample.select_dtypes(exclude="number").columns[0])] = "BRAND_NEW_CATEGORY"
print(reloaded.predict_proba(sample)[:, 1].round(3))

## 9. Summary & findings
* **Split first:** stratified 80/20 split performed before any fitting; the leakage check in Section 5 asserts that the scaler statistics equal the *training* statistics.
* **Imputation:** numeric columns → median, categorical columns → most frequent, both fitted on train only.
* **Encoding & scaling:** `StandardScaler` for numeric features and `OneHotEncoder(handle_unknown="ignore")` for categoricals, combined with `ColumnTransformer`.
* **Correlation:** redundant numeric pairs (e.g. tenure / monthly charges / total charges) are flagged in Section 6; tree models tolerate them, linear models may benefit from dropping one.
* **Importance:** mutual information and random-forest rankings largely agree on the strongest churn drivers (e.g. contract type, tenure, charges); the bottom-quartile features are candidates for removal.
* **Reusability:** `build_preprocessor()` and `make_full_pipeline()` work on any tabular DataFrame, and the fitted pipeline is exported as a `.joblib` file.

**Limitations / next steps (Task 04):** tune hyper-parameters, compare XGBoost/LightGBM, and check calibration and fairness across customer segments from the Task 02 data card.